In [ ]:
!pip install supervision ultralytics tqdm -q
import supervision as sv
import cv2
import pandas as pd
from tqdm import tqdm

from google.colab import drive
from google.colab.patches import cv2_imshow

import matplotlib.pyplot as plt
import numpy as np
drive.mount("/content/drive")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# CREAR DATASET DE POSICION DE PELOTA EN TODO EL VIDEO

#### No sirve ByteTrack para rastrear la peltoa, por alguna razon

## 0. Funciones para detectar la pelota

In [ ]:
def BGR2HSV(imagen):
    return cv2.cvtColor(imagen,cv2.COLOR_BGR2HSV)


#ESTA FUNCION RECIBE UNA IMAGEN, OBTIENE LA MASCARA DE HSV
#APLICA MODIFICACIONES MORFOLOGICAS

def procesarMascara(imagen_bgr):
    imagen = BGR2HSV(imagen_bgr)
    naranja_inferior = np.array([5,160,200])
    naranja_superior = np.array([20,202,255])

    mascaraNormal = cv2.inRange(imagen,naranja_inferior,naranja_superior)
    mascaraProcesada = cv2.inRange(imagen,naranja_inferior,naranja_superior)

    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15)) #Para cerrar espacios vacios de la mascara
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3)) #Para abrir espacios, quitar granos de arena y ruido

    #Por el momento solo vamos a cerrar espacios
    mascaraProcesada = cv2.morphologyEx(mascaraProcesada,cv2.MORPH_CLOSE,kernel_close)

    return mascaraNormal,mascaraProcesada

def detectar_pelota(imagen_bgr:np.ndarray,min_area:int = 1,max_area:int = 1000) -> sv.Detections:
    #imagen = BGR2HSV(imagen_bgr)
    mascara_pelota = procesarMascara(imagen_bgr)[1]
    #cv2_imshow(mascara_pelota)
    contornos,valor = cv2.findContours(mascara_pelota,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)

    lista_xyxy =[]
    lista_clase_ids =[]
    lista_tracker_ids = []
    datos = {
        'class_name' : [],
        'area_pixeles':[]
    }

    for cont in contornos:
        area = cv2.contourArea(cont)
        if(area>max_area or area<min_area):
            continue
        x,y,ancho,alto = cv2.boundingRect(cont)
        lista_xyxy.append([x,y,x+ancho,y+alto])
        lista_clase_ids.append(99999)
        nombre_clase = "orange ball"
        datos['class_name'].append(nombre_clase)
        datos['area_pixeles'].append(area)
        lista_tracker_ids.append(-1)

    if not lista_xyxy:
        return sv.Detections.empty()

    detecciones = sv.Detections(
        xyxy = np.array(lista_xyxy,dtype=np.float32),
        class_id = np.array(lista_clase_ids,dtype=int),
        tracker_id = np.array(lista_tracker_ids,dtype=int),
        data = datos
    )

    return detecciones


def proyectar_puntos(detecciones,matriz):
    #Esta funcion recibe un objeto de detecciones y los manda al espacio canonico
    #Devueve una lista de puntos canonicos
    #Proyecta solo el CENTRO DE LA CAJA

    cajas = detecciones.xyxy
    lista_x=[]
    lista_y=[]

    for caja in cajas:
        x1=caja[0];y1=caja[1]
        x2=caja[2];y2=caja[3]
        x = (x1+x2)/2
        y = (y1+y2)/2

        punto = np.float32([x,y])
        proyeccion = cv2.perspectiveTransform(punto.reshape(1,1,2),matriz)
        lista_x.append(int(proyeccion[0][0][0]))
        lista_y.append(int(proyeccion[0][0][1]))

        #print()

    return lista_x,lista_y


H = np.array([
    [1.14741274, -0.180942021, 401.053893],
    [0.136897422, 1.14115349, -258.833614],
    [-0.000017149164, -0.0000341202393, 1.0]
])

## 1.Procesar el video con cv2.videoCapture

No voy a usar trackers ni anotaciones, solo voy a detectar la posicion de la pelota y guardare los datos. Tambien filtrare todo lo posible para tener un dataset lo mas limpio posible.

In [ ]:
#inicializar diccionario y capturador
ruta_video = "/content/drive/MyDrive/PROYECTO SAM 3/VIDEOS/IMG_9938.MOV"

datos = {
    "frame":[],
    "tracker_id":[],
    "class_name":[],
    "x_canon":[],
    "y_canon":[],
    "num_detecciones":[],
    "area_pixeles":[],
}

#INICIALIZAR EL CAPTURADOR

cap = cv2.VideoCapture(ruta_video)

fps_original = cap.get(cv2.CAP_PROP_FPS)
ancho_original = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto_original = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frames_totales = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))


numero_frame = 1

with tqdm(total=frames_totales,desc="Procesando video",unit="frame") as pbar:
    while cap.isOpened():
        ret,frame = cap.read()
        if not ret:
            print("NOT RET")
            break

        detecciones = detectar_pelota(frame,60,850)
        n = len(detecciones)

        if (n>0):
            lista_x_canon,lista_y_canon = proyectar_puntos(detecciones,H)
            escala = 8

            lista_x_real = [x/escala for x in lista_x_canon]
            lista_y_real = [y/escala for y in lista_y_canon]

            datos["frame"] = datos["frame"] + [numero_frame]*n
            datos['tracker_id'] = datos["tracker_id"] + detecciones.tracker_id.tolist()
            datos["class_name"] = datos["class_name"] + detecciones.data["class_name"]
            datos["x_canon"] = datos["x_canon"] + lista_x_real
            datos["y_canon"] = datos["y_canon"] + lista_y_real
            datos["num_detecciones"] = datos["num_detecciones"] +[n]*n
            datos["area_pixeles"] = datos["area_pixeles"] +detecciones.data["area_pixeles"]

        else:
            datos["frame"] = datos["frame"] + [numero_frame]
            datos['tracker_id'] = datos["tracker_id"] + [None]
            datos["class_name"] = datos["class_name"] + [None]
            datos["x_canon"] = datos["x_canon"] + [None]
            datos["y_canon"] = datos["y_canon"] + [None]
            datos["num_detecciones"] = datos["num_detecciones"] + [n]
            datos["area_pixeles"] = datos["area_pixeles"] +[None]

        numero_frame = numero_frame+1
        pbar.update(1)


        #Lo siguiente es nadamas para verificar que la cantidad vaya siendo la misma en todos
        #print("Len frames: ",len(datos['frame']))
        #print("Len tracker_id: ",len(datos['tracker_id']))
        #print("Len class_name: ",len(datos['class_name']))
        #print("Len x_canon: ",len(datos['x_canon']))
        #print("Len y_canon: ",len(datos['y_canon']))
        #print("Len num_detecciones: ",len(datos['num_detecciones']))
        #print("Len area_pixeles: ",len(datos['area_pixeles']))


cap.release()

Procesando video: 100%|█████████▉| 19393/19407 [16:51<00:00, 19.18frame/s]

NOT RET


In [ ]:
cap.release()

## 2. Guardar los datos como un dataframe de pandas

In [ ]:
#Lo siguiente es nadamas para verificar que la cantidad vaya siendo la misma en todos
print("Len frames: ",len(datos['frame']))
print("Len tracker_id: ",len(datos['tracker_id']))
print("Len class_name: ",len(datos['class_name']))
print("Len x_canon: ",len(datos['x_canon']))
print("Len y_canon: ",len(datos['y_canon']))
print("Len num_detecciones: ",len(datos['num_detecciones']))
print("Len area_pixeles: ",len(datos['area_pixeles']))


Len frames:  20418
Len tracker_id:  20418
Len class_name:  20418
Len x_canon:  20418
Len y_canon:  20418
Len num_detecciones:  20418
Len area_pixeles:  20418


In [ ]:
ruta_guardado = "/content/drive/MyDrive/PROYECTO SAM 3/datasets/pelota_video_completo/datosPelotaVideoCompleto.csv"
datos_pelota = pd.DataFrame(datos)
datos_pelota.to_csv(ruta_guardado,index=False,sep=",", encoding="utf-8-sig")